In [1]:
%load_ext autoreload

In [2]:

import pandas as pd
import json
import sklearn
import glob
import pickle
from sklearn.model_selection import train_test_split
from collections import Counter


pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
%autoreload
import sys
sys.path.insert(0, '../../style_generation_pipeline')

from data import *
from cluster_representation import *

/mnt/swordfish-pool2/milad/conda-envs/gpu-env/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-03-28 06:31:16.080354: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743157876.505775   42476 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743157876.661260   42476 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-28 06:31:18.114805: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimize

In [4]:
path='/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/'

### Build Clusters Style Representation

In [6]:
#llm_style_feats = path + '/refined_and_aggregated_features_final.csv'
#llm_style_feats = path + '/refined_and_aggregated_features_final_manually_processed.csv'
llm_style_feats = path + 'refined_and_aggregated_features_final_manually_processed_w_high_and_verifiable_feats.csv'
llm_and_g2v_style_feats = path + '/llm_and_gram2vec_feats.csv'
g2v_style_feats = path + '/gram2vec_feats.csv'


In [7]:
# df = pd.read_csv(llm_style_feats)
# df[['shortend_attribute_name.v2', 'original_attribute_name']].drop_duplicates().to_csv('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/original_feature_name.csv')

In [8]:
def build_cluster_representation(clustering_path, output_path, top_k=10, summarize_with_gpt=False):
    #feat_clm = 'final_attribute_name'
    styles_corpus_path = llm_and_g2v_style_feats
    feat_clm = 'final_attribute_name_manually_processed'

    # We are not using the featus_dict for now
    #df = pd.read_csv(llm_style_feats_dict)
    #feats_dict = {x[0]: (x[1], x[2]) for x in zip(df[feat_clm].tolist(), df.Level.tolist(), df.Verifiability.tolist())}
    
    #Representative Summarization
    clusters_tfidf_rep_df = generate_interpretable_space_representation(clustering_path, styles_corpus_path, feat_clm, 'tfidf_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)
    #Contrastive Summarization
    #clusters_contra_rep_df = generate_interpretable_space_contra_representation(clustering_path, styles_corpus_path, feat_clm, 'con_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)

    feat_clm = 'gram2vec_feats'
    styles_corpus_path = g2v_style_feats
    #Representative Summarization
    clusters_tfidf_rep_g2v_df = generate_interpretable_space_representation(clustering_path, styles_corpus_path, feat_clm, 'tfidf_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)
    #Contrastive Summarization
    #clusters_contra_rep_g2v_df = generate_interpretable_space_contra_representation(clustering_path, styles_corpus_path, feat_clm, 'con_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)

    #clusters_tfidf_rep_df['llm_con_rep'] = clusters_contra_rep_df['con_rep']
    clusters_tfidf_rep_df['llm_tfidf_rep'] = clusters_tfidf_rep_df['tfidf_rep']
    clusters_tfidf_rep_df['llm_tfidf_weights'] = clusters_tfidf_rep_df['tfidf_rep_dist']
    clusters_tfidf_rep_df['g2v_tfidf_rep'] = clusters_tfidf_rep_g2v_df['tfidf_rep']
    clusters_tfidf_rep_df['g2v_tfidf_weights'] = clusters_tfidf_rep_g2v_df['tfidf_rep_dist']
    #clusters_tfidf_rep_df['g2v_con_rep'] = clusters_contra_rep_g2v_df['con_rep']

    clusters_tfidf_rep_df[['cluster_label', 'llm_tfidf_rep', 'llm_tfidf_weights', 'g2v_tfidf_rep', 'g2v_tfidf_weights']].to_json(output_path)
    
    return clusters_tfidf_rep_df

In [24]:
build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interpretable_space_representations.json', top_k=10)

In [30]:
build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interp_space_148_clusters/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interp_space_148_clusters/interpretable_space_representations.json', top_k=10)

In [122]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/interpretable_space_representations.json', top_k=10, summarize_with_gpt=True)

Number of style feats  719


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai

Number of style feats  448


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and the results were backed up. 💾


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and the results were backed up. 💾
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


In [30]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters/interpretable_space_representations.json', top_k=10, summarize_with_gpt=True)

Number of style feats  853


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


Number of style feats  448


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and the results were backed up. 💾


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and the results were backed up. 💾
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


In [9]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_03/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_03/interpretable_space_representations.json', top_k=10, summarize_with_gpt=True)

Number of style feats  853


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


Number of style feats  448


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


In [9]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_09/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_09/interpretable_space_representations.json', top_k=10, summarize_with_gpt=True)

Number of style feats  853


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and the results were backed up. 💾
[ 🤖 DataDreamer 💤 ] Step 'sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and the results were backed up. 💾
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summariz

Number of style feats  448


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and the results were backed up. 💾


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and the results were backed up. 💾
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


In [23]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_28/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_28/interpretable_space_representations.json', top_k=10, summarize_with_gpt=False)

Number of style feats  853
Number of style feats  448


In [11]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_12/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters_12/interpretable_space_representations.json', top_k=10, summarize_with_gpt=False)

In [9]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_clusters_035/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_clusters_035/interpretable_space_representations.json', top_k=10, summarize_with_gpt=False)

Number of style feats  853
Number of style feats  448


In [10]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_clusters_07/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_clusters_07/interpretable_space_representations.json', top_k=10, summarize_with_gpt=False)

Number of style feats  853
Number of style feats  448


In [12]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_clusters_09/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_clusters_09/interpretable_space_representations.json', top_k=10, summarize_with_gpt=False)

Number of style feats  853
Number of style feats  448


In [147]:
# Converting gra2vec style corpus from jsonl to csv
# g2v_style_corpus = pd.read_json('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/normalized_all_document_gram2vec_top_features.jsonl', lines=True)
# g2v_style_corpus = g2v_style_corpus.explode('gram2vec_feats')
# g2v_style_corpus.to_csv('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/normalized_all_document_gram2vec_top_features.csv', index=False)